# kNN y Naive Bayes

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import style

import seaborn as sns

import missingno as msno

%matplotlib inline

In [ ]:
import warnings
warnings.filterwarnings("ignore")

style.use('ggplot') or plt.style.use('ggplot')

In [ ]:
def graf_histo_box_numericas(dataframe, var_num):
    """
    Realiza graficas histograma y boxplot de variables numéricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_num: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """

    TABLEAU_CMP = ('tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', \
                   'tab:gray','tab:olive', 'tab:cyan')

    fig, axes = plt.subplots(len(var_num), 2, \
                             figsize=(20, 5 * len(var_num)), \
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.1})
    ax = axes.ravel()

    # graficas distribucion y boxplot de cada atributo
    for idx, atributo in enumerate(var_num):

        # distribucion (histograma)
        sns.histplot(dataframe[atributo], bins=30, ax=ax[2 * idx], \
                     color=TABLEAU_CMP[idx % len(TABLEAU_CMP)], \
                     alpha=0.15)

        # titulo, etiquetas histograma
        ax[2 * idx].set_title(f'HISTOGRAMA {atributo}')
        ax[2 * idx].set_xlabel(f'Valores {atributo}')
        ax[2 * idx].set_ylabel("Frequencia")

        # boxplot
        sns.boxplot(x=atributo, data=dataframe, ax=ax[2 * idx + 1], color=TABLEAU_CMP[idx % len(TABLEAU_CMP)])

        # titulo, etiquetas boxplot
        ax[2 * idx + 1].set_title(f'BOXPLOT {atributo}')
        ax[2 * idx + 1].set_xlabel(f'Valores {atributo}')

## Carga del dataset

Vamos a cargar el dataset de "iris". Scikit Learn disponde de varios [datasets](https://scikit-learn.org/stable/datasets/toy_dataset.html) útiles para el aprendizaje:

In [ ]:
# scikit proporciona datasets
from sklearn.datasets import load_iris


In [ ]:
load_iris().keys()

In [ ]:
# devuelve atributos(X) y target (y) del dataframe
X, y = load_iris(return_X_y=True, as_frame=True)

# si lo necesito como DF
iris_df = X.copy()
iris_df['especie_iris'] = y

iris_df.head()

\

El **target** consiste en las 3 clases de flores iris:

In [ ]:
load_iris().target_names

Los **atributos** son:

In [ ]:
load_iris().feature_names

Formamos los nombre de las columnas:

In [ ]:
# "arreglamos" nombres columnas atributos
columnas = [columna[:-5].replace(" ","_") for columna in iris_df.columns.to_list()]

columnas

In [ ]:
iris_df.columns = columnas

iris_df.head()

Echemos un vistazo breve:

In [ ]:
iris_df.info()

\

Los **valores del target (numéricos)** se relacionan con el **orden del campo "target_names"**.

In [ ]:
iris_df.especie.value_counts()

In [ ]:
load_iris().target_names

In [ ]:
iris_df.especie.value_counts()


In [ ]:
iris_df.especie = iris_df.especie.replace({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

iris_df.especie.value_counts()

\

## Explorar dataframe

In [ ]:
iris_df.head()

In [ ]:
# columna target ordenada.

iris_df.sample(10)

In [ ]:
iris_df.describe()

\

### Primeras gráficas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline


**TARGET**

In [ ]:
# SEABORN

sns.countplot(x='especie', data=iris_df,palette='Set2')

**ATRIBUTOS**

In [ ]:
graf_histo_box_numericas(iris_df, ['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])

\

Por cuestiones didácticas y a la hora de **visualizar las fronteras de decisión en la clasificación para los distintos modelos (para comparar los modelos)**, vamos a buscar las 2 características que me permitan diferenciar más las clases (especies):

In [ ]:
sns.pairplot(data=iris_df, hue='especie', palette={'setosa': 'red', 'versicolor': 'green', 'virginica': 'blue'})

**Podemos concluir que a la hora de elegir 2, son los atributos de los pétalos en donde se distinguen (se separan) más las clases**.

Las utilizaremos para comparar el desempeño de modelos y sacar conclusiones.

In [ ]:
iris_df.columns

\

## División TRAIN - TEST

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
iris_df.head()

In [ ]:
# Separamos atributos y target:

X = iris_df.drop(columns='especie')

y = iris_df.especie

X.shape, y.shape

**NOTA**: scikit learn **no trabaja con campos tipo texto**. La libreria proporciona "transformadores" para **codificar a numérico** (lo veremos más adelante).

Por ahora nos valdremos de usar método "map" sobre la "y":

In [ ]:
load_iris().target_names

In [ ]:
y = y.map({'setosa': 0, 'versicolor': 1, 'virginica':2})

y.value_counts()

Ahora si podemos pasar a la división TRAIN-TEST:

In [ ]:
# division TRAIN-TEST

# queremos que haya la misma proporción de clases en TRAIN y TEST ===> parámetro "stratify"

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)


In [ ]:
X_train.shape, X_test.shape

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

\

# k-Nearest Neighbours

## Entrenamiento. Predicción . Evaluación.

Para el parámetro más influyente de largo, el nº de vecinos (k), vamos primero a partir de valor (1), modelo complejo, e iremos incrementando valor para ver las distintas fases (entrenamiento, predicción, evaluación métrica por defecto)

In [ ]:
# Instanciamos (creamos) el modelo:

from sklearn.neighbors import KNeighborsClassifier

clasif = KNeighborsClassifier(n_neighbors=1)



In [ ]:
# lo entrenamos con la partición TRAIN

clasif.fit(X_train, y_train)

El **método "score"** utiliza como **métrica por defecto para clasificación "accuracy" (proporción de acierto)**.

**Una vez entrenado el modelo, lo evaluamos con la partición "test"**

**NOTA**: muchas veces se hace también con la partición TRAIN para comparar ambas y detectar señales de overfittting o underfitting

In [ ]:
clasif.score(X_test, y_test)


In [ ]:
clasif.score(X_train, y_train)

**Que conclusiones podemos sacar?**

**Probamos con 3 vecinos, ¿ hacia donde nos movemos?**

In [ ]:
clasif3 = KNeighborsClassifier(n_neighbors=3)

clasif3.fit(X_train, y_train)

In [ ]:
clasif3.score(X_train,y_train)

In [ ]:
clasif3.score(X_test,y_test)

Podemos **predecir de que tipo de flor** se trata si usamos el **método "predict"** y **le pasamos los atributos de la flor en cuestión**:



In [ ]:
atributos_flor = np.array([[4.6, 3.1, 1.5, 0.2]])

clasif.predict(atributos_flor)

In [ ]:
clasif3.predict(atributos_flor)



In [ ]:
mi_prediccion = clasif3.predict(atributos_flor)

load_iris().target_names[mi_prediccion[0]]

\

Podemos comparar las predicciones para X_test con los valores reales (y_test) y volver a calcular el "accuracy":

In [ ]:
clasif3.predict(X_test) == y_test

In [ ]:
comprobar = clasif.predict(X_test) == y_test

comprobar.mean()

## Comparación de modelos según "k". Fronteras de decisión

Nos vamos a valer de unas gráficas donde se visualizarán las fronteras de decisión para ver la influencia del parámetro "k".

Para ello nos va a hacer falta la siguiente **función** (**basta con utilizarla**):

In [ ]:
from matplotlib.colors import ListedColormap


# Definimos la función que nos graficará las fronteras de decisión

def plot_decision_boundaries(model, X, y, delta: float = .02) -> None:
    """Plot data points and deicision boundaries learned by the model.

    Arguments:
    ----------
    model: scikit-learn like model

    X: np.array[n_samples, n_features]
        Only first 2 features will be considered because it is a 2d plot.
        Feature 0 in the x axis, and feature 1 in the y axis.

    y: np.array
        Labels for each sample.

    delta: float
        Increment between consecutive points when computing the grid for plotting boundaries.
        Lower value for higher resolution.
    """

    # Creamos la meshgrid con los valores mínimo y máximo de 'x' i 'y'.
    # La variable X es nuestro dataframe con las variables a estudiar (las del pétalo o las del sépalo)
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(np.arange(x_min, x_max, delta),
                         np.arange(y_min, y_max, delta))

    #Predecimos el clasificador con los valores de la meshgrid
    # En este caso model será nuestra variable que contiene el modelo a estudiar, es decir K-nn, SVM,...
    # Por ejemplo para K-nn sería model = KNeighborsClassifier()

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])

    # Creamos mapas de colores con ListedColormap para ver como separa las clases.
    # En este caso usaremos:
    # Iris-setosa : darkorange
    # Iris-versicolor: c
    # Iris-virginica: darkblue

    cmap_light = ListedColormap(['orange', 'cyan', 'cornflowerblue'])
    cmap_bold = ListedColormap(['darkorange', 'c', 'darkblue'])

    # Ponemos el resultado en una figura de color
    Z = Z.reshape(xx.shape)
    plt.figure()
    plt.pcolormesh(xx, yy, Z, cmap= cmap_light)

    # Dibujamos también los puntos de entrenamiento
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap= cmap_bold)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.show()

Para poder visualizarla en una gráfica 2D, elegiremos sólo 2 de los atributos. Si recuerdan los mejores atributos que "separaban" las clases eran las del pétalo ("petal_length", "petal_width"):

In [ ]:
X.columns

In [ ]:
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X.iloc[:,2:], y, test_size=0.3,stratify=y, random_state=42)

In [ ]:

k = [1,3,5,7]

modelos = []

for vecinos in k:

    clasif = KNeighborsClassifier(n_neighbors=vecinos)
    clasif.fit(X_train_f, y_train_f)

    modelos.append(clasif)


In [ ]:
modelos[2]

Veamos las gráficas:

In [ ]:
len(modelos)

**k = 1**

In [ ]:
plot_decision_boundaries(modelos[0], X_train_f.values, y_train_f.values)

\

**k = 3**

In [ ]:
plot_decision_boundaries(modelos[1], X_train_f.values, y_train_f.values)

\

**k = 5**

In [ ]:
plot_decision_boundaries(modelos[2], X_train_f.values, y_train_f.values)

\

**k=7**

In [ ]:
plot_decision_boundaries(modelos[3], X_train_f.values, y_train_f.values)

Vemos como **según aumenta "k" obteniendo modelos menos complejos, la segunda frontera de decisión** (la que vale la pena prestar atención) **se suaviza**.

\

# Naive Bayes

**NOTA**: no es necesario volver a hacer la partición TRAIN-TEST.

## Entrenamiento. Predicción . Evaluación.

Dadas las características de los datos que tenemos (**atributos continuos**) vamos a utilizar el modelo **"GaussianNB"**.

Para este modelo jugaremos con varios valores para el **parámetro "var_smoothing"** (proporción, entre cero y uno). **Según aumentamos** su valor vamos consiguiendo modelos menos complejos.

In [ ]:
# Instanciamos (creamos) el modelo:

from sklearn.naive_bayes import GaussianNB

clasif = GaussianNB(var_smoothing=0.33)



In [ ]:
# lo entrenamos con la partición TRAIN

clasif.fit(X_train, y_train)

In [ ]:
clasif.score(X_test, y_test)


In [ ]:
clasif.score(X_train, y_train)

In [ ]:
# Instanciamos (creamos) el modelo:

from sklearn.naive_bayes import GaussianNB

clasif = GaussianNB(var_smoothing=1)



In [ ]:
# lo entrenamos con la partición TRAIN

clasif.fit(X_train, y_train)

In [ ]:
clasif.score(X_test, y_test)


In [ ]:
clasif.score(X_train, y_train)

En este caso nos movemos en un rango muy pequeño y como se puede observar solo vemos diferencias yendonos a los extremos.Comprobamos que **si vamos utlizando modelos menos complejos** nos acercamos al valor de "var_smoothing" para el mejor desempeño (acc = 0.90 aprox.), si bien las diferencias no son grandes.

Pasamos a la **predicción**:



In [ ]:
atributos_flor = np.array([[4.6, 3.1, 1.5, 0.2]])

clasif.predict(atributos_flor)

In [ ]:
load_iris().target_names[0]

\

De nuevo comparamos las predicciones para X_test con los valores reales (y_test) para volver a calcular el "accuracy":

In [ ]:
clasif.predict(X_test) == y_test

In [ ]:
comprobar = clasif.predict(X_test) == y_test

comprobar.mean()

## Comparación de modelos según "var_smoothing". Fronteras de decisión

Visualizaremos las fronteras de decisión para ver la influencia del **parámetro "var_smoothing"**, usando la misma función para dibujarlas que con el modelo anterior "KNN"


Recordar que volveremos a utilizar sólo 2 de los atributos ("petal_length", "petal_width"), y no es necesario volver a calcular la partición opara este conjunto reducido:

In [ ]:

var_sm = [1e-9, 0.33]

modelos = []

for v in var_sm:

    clasif = GaussianNB(var_smoothing=v)
    clasif.fit(X_train_f, y_train_f)

    modelos.append(clasif)



Veamos las gráficas:

**var_smoothing = 1e-9**

In [ ]:
plot_decision_boundaries(modelos[0], X_train_f.values, y_train_f.values)

\

**var_smoothing = 0.33**

In [ ]:
plot_decision_boundaries(modelos[1], X_train_f.values, y_train_f.values)

\

Vemos como **según aumenta "var_smoothing" obtenemos modelos menos complejos, con ambas fronteras de decisión** **muy suavizadas**.

\

# MANOS A LA OBRA:

El dataset **"wine"** es un dataset que recoge **multiples medidas de los niveles de sustancias químicas presentes en 3 clases de vino** (class_0,class_1, class_2).

**Todos los atributos son numéricos y podeis dejar el target como está (numérico)**

In [ ]:
from sklearn.datasets import load_wine

X, y = load_wine(return_X_y=True, as_frame=True)

type(X), type(y)

In [ ]:
X.shape, y.shape

In [ ]:
X.head()

In [ ]:
X.columns

In [ ]:
y.unique()

In [ ]:
y.value_counts()

In [ ]:
load_wine().keys()

In [ ]:
load_wine().target_names

\

## EJERCICIO:

1.- "Construir" el dataframe con atributos y target.

2.- Realizar la partición TRAIN-TEST

3.- Entrenar modelos "k-NN" para al menos 2 valores de "k". Comparar los valores de "accuracy" (TRAIN y TEST) entre modelos

4.- Entrenar modelos "NB" para al menos 2 valores de "var_smoothing". Comparar los valores de "accuracy" (TRAIN y TEST) entre modelos.

**NO ES NECESARIO** dibujar las fronteras de decisión